# 01 · Data Cleaning

**Plain English:** raw sales data is messy: duplicate rows, cancelled orders, free samples and
guest checkouts. Before we can trust any revenue number we remove those rows and record
exactly how many each rule removed.

**Technical:** this notebook only *calls* `src.data.clean.clean_transactions`, the same function the
pipeline uses, so the numbers here match the database and dashboard.

In [1]:
import sys; sys.path.insert(0, "..")
import plotly.io as pio
pio.renderers.default = "png"  # static images so charts render on GitHub
import kaleido; kaleido.start_sync_server(silence_warnings=True)  # one shared browser for reliable PNG export
import logging, warnings
logging.getLogger("kaleido").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="The kopts argument")

import pandas as pd
from src.data.ingest import load_raw
from src.data.clean import clean_transactions

## 1. Load the raw data
UCI *Online Retail II*: every line item sold by a UK online gift wholesaler, Dec 2009 to Dec 2011.
`load_raw` stacks both yearly Excel sheets (cached as Parquet for speed).

In [2]:
raw = load_raw()
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"Dates: {raw['InvoiceDate'].min():%Y-%m-%d} to {raw['InvoiceDate'].max():%Y-%m-%d}")
raw.head()

1,067,371 rows x 8 columns
Dates: 2009-12-01 to 2011-12-09


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 2. Column types and missing values

In [3]:
pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_%": (raw.isna().mean() * 100).round(2),
})

,dtype,missing,missing_%
Invoice,str,0,0.00
StockCode,str,0,0.00
Description,str,4382,0.41
Quantity,int64,0,0.00
InvoiceDate,datetime64[us],0,0.00
Price,float64,0,0.00
Customer ID,float64,243007,22.77
Country,str,0,0.00


Missing `Customer ID` is the big gap: those are guest checkouts we can't link to a person.
Negative quantities hint at returns; we'll handle them next.

## 3. Apply the cleaning rules
Rules are applied **in order**, and each row is counted only under the first rule that removes it,
so the counts add up exactly.

In [4]:
clean, report = clean_transactions(raw)

WHY = {
    "exact_duplicates": "The two yearly sheets overlap (Dec 2010), so some rows appear twice.",
    "cancellations_and_adjustments": "Invoices starting with 'C' are returns; 'A' are bad-debt write-offs, not sales.",
    "non_positive_quantity": "Zero/negative quantities are stock corrections, not purchases.",
    "non_positive_price": "Free items or price errors would distort revenue and order values.",
    "non_product_codes": "Postage, bank charges, Amazon fees and manual adjustments are not products.",
    "missing_customer_id": "Customer-level analysis (RFM, clustering, prediction) needs an ID.",
}
rules = pd.DataFrame(
    [(rule, report[rule], WHY[rule]) for rule in WHY],
    columns=["rule", "rows_removed", "why"],
)
rules["%_of_raw"] = (rules["rows_removed"] / report["raw_rows"] * 100).round(2)
rules

2026-11-17 21:43:40,459 | INFO | src.data.clean | Cleaning kept 770524 of 1067371 rows


,rule,rows_removed,why,%_of_raw
0,exact_duplicates,34335,"The two yearly sheets overlap (Dec 2010), so s...",3.22
1,cancellations_and_adjustments,19110,Invoices starting with 'C' are returns; 'A' ar...,1.79
2,non_positive_quantity,3393,Zero/negative quantities are stock corrections...,0.32
3,non_positive_price,2619,Free items or price errors would distort reven...,0.25
4,non_product_codes,4343,"Postage, bank charges, Amazon fees and manual ...",0.41
5,missing_customer_id,226838,"Customer-level analysis (RFM, clustering, pred...",21.25


## 4. Result

In [5]:
summary = pd.Series({k: report[k] for k in ["raw_rows", "clean_rows", "customers", "invoices"]})
print(summary.map("{:,}".format).to_string())
print(f"\nKept {report['clean_rows'] / report['raw_rows']:.1%} of raw rows.")
clean.head()

raw_rows      1,067,371
clean_rows      770,524
customers         5,839
invoices         36,332

Kept 72.2% of raw rows.


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


`clean_transactions` also runs `validate_clean`, which raises an error if any contract is broken
(positive quantity and price, an ID on every row, no cancellations, no duplicates). It passed,
so the table above is safe to load into SQLite for the analysis notebooks.